This prototype demonstrates **XCharge+**, an AI-powered driving companion designed to assist electric vehicle (EV) drivers in planning trips and managing battery usage. The system analyzes key factors such as vehicle battery capacity, trip distance, driving environment, and driver behavior to estimate energy consumption and predict the vehicle’s state of charge upon arrival. Based on this analysis, the model determines whether the driver can safely complete the trip or if charging is recommended along the way. The application also references a dataset of EV charging stations to suggest available charging locations near the destination province.

In addition to its analytical component, the system incorporates a Generative AI layer through persona-based responses, where different assistant personalities present the trip insights in conversational formats. Personas such as BOSS, Sandy, Carlo, Jam, and Manu provide the same analytical insights but communicate them in distinct tones to simulate a personalized driving assistant experience. This demonstrates how GenAI can transform structured transportation data into a more engaging and human-centered interface. For demonstration purposes, the prototype assumes clear weather conditions and simplified charger selection, as a real-world implementation would integrate APIs for live weather, mapping, and charging network data. Overall, the prototype illustrates how AI can function as an intelligent EV companion that reduces range anxiety and supports better driving decisions.

**Block 1: Installation of Libraries**



In [1]:
!pip install pandas numpy openpyxl

**Block 2: Import Libraries**

In [2]:
import pandas as pd
import numpy as np
import math

**Block 3: Dataset upload**

The following datasets were uploaded:


*   EV charging patterns
*   EV charging stations
*   User trip data
*   User Car specifications





In [3]:
from google.colab import files
uploaded = files.upload()

Saving ev_charging_patterns.csv to ev_charging_patterns.csv
Saving ev_charging_stations.csv to ev_charging_stations.csv
Saving Trip_data.csv to Trip_data.csv
Saving W5_car_specs.csv to W5_car_specs.csv


**Block 4: Dataset Loading**

In [6]:
charging_stations = pd.read_csv('ev_charging_stations.csv')

charging_patterns = pd.read_csv('ev_charging_patterns.csv')

trip_data = pd.read_csv('Trip_data.csv')

w5_specs = pd.read_csv('W5_car_specs.csv')

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


**Block 5: Defining of EV constants**

We define the core parameters of the EV used. These inmclude the battery capacity of the Weltmeister W5, the base energy efficenciy, safety buffer. Constants are used to estimate battery consumptions and trip feasibility

In [34]:
BATTERY_KWH = 52
ARRIVAL_BUFFER = 20
BASE_EFFICIENCY = 7.0  # km per kWh ideal efficiency

print("Vehicle: Weltmeister W5")
print("Battery capacity:", BATTERY_KWH, "kWh")

Vehicle: Weltmeister W5
Battery capacity: 52 kWh


**Block 6: Driver Behavior and Environment**

Code adjusts vehicle efficiency based on user's driving conditons. Factors like road landscape and aggressive driving behavior can impact how much energy the vehicle consumes. This code simulates those real-world conditons by modifying the baseline efficiency

In [10]:
def adjust_efficiency(base_efficiency, environment, weather="none", aggressive_driver=True):

    efficiency = base_efficiency

    if environment.lower() == "city":
        efficiency *= 1.10

    elif environment.lower() == "highway":
        efficiency *= 0.85

    elif environment.lower() == "mixed":
        efficiency *= 1.00

    elif environment.lower() == "mountain":
        efficiency *= 0.50

    if weather.lower() == "rain":
        efficiency *= 0.95

    elif weather.lower() == "heavy rain":
        efficiency *= 0.90

    if aggressive_driver:
        efficiency *= 0.75

    return efficiency

**Block 7: Battery Prediction**

Code estimates energy consumption during a trip. It calculates how much energy is required for each trip, how much battery percentage is consumed and the projected battery level upon arrival. Code also determines whether charging is recommended prior to a trip

In [35]:
def predict_trip(distance_km, soc, environment, weather="none"):

    efficiency = adjust_efficiency(BASE_EFFICIENCY, environment, weather)

    energy_needed = distance_km / efficiency

    soc_used = (energy_needed / BATTERY_KWH) * 100

    projected_arrival_soc = soc - soc_used

    soc_required_with_buffer = soc_used + ARRIVAL_BUFFER

    charging_needed = soc < soc_required_with_buffer

    return {
        "adjusted_efficiency": round(efficiency, 2),
        "energy_needed_kwh": round(energy_needed, 2),
        "soc_used": round(soc_used, 2),
        "projected_arrival_soc": round(projected_arrival_soc, 2),
        "soc_required_with_buffer": round(soc_required_with_buffer, 2),
        "charging_needed": charging_needed
    }

**Block 8:Distance Calculation**



In [36]:
def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2

    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1-a))

**Block 9: Charging Station Recommendation**

Dataset does not include geographic coordinagtes so the prototype filters stations based on destination. The code returns relevant charging stations that the driver can visit if charging is required.

In [29]:
def recommend_chargers(location_province, stations_df, top_n=3):

    df = stations_df.copy()

    df["Province"] = df["Province"].astype(str).str.lower().str.strip()
    location_province = str(location_province).lower().strip()

    filtered = df[df["Province"].str.contains(location_province, na=False)]

    if len(filtered) == 0:
        filtered = df

    return filtered.head(top_n)

**Block 10: Persona Generator**

This code generates conversational responses based on the selected persona. Different personalities provide trip insights in varying tones while delivering the same analytical and predictive insights.

In [37]:
def persona_response(persona, destination, result, recommended_station=None):

    intro = ""

    if persona.lower() == "boss":
        intro = "System analysis complete."

    elif persona.lower() == "sandy":
        intro = "Rise and conquer, you little ray of sunshine."

    elif persona.lower() == "carlo":
        intro = "Fun fact: EVs convert most electricity directly into motion."

    elif persona.lower() == "jam":
        intro = "You are fine."

    elif persona.lower() == "manu":
        intro = "Let's think this through."

    recommendation = "Proceed with the trip."

    if result["charging_needed"]:
        recommendation = "Charging is recommended before or during the trip."

    station_text = ""

    if recommended_station is not None:
        station_text = f"Recommended charging station: {recommended_station['Charging Station Name']} ({recommended_station['City']}, {recommended_station['Province']})"

    disclaimer = """
Demo Disclaimer:
This prototype assumes clear weather conditions (no rain).
Future versions would integrate real-time weather APIs.
Results may vary during rainy conditions.
"""

    response = f"""
{intro}

Destination: {destination}

Adjusted efficiency: {result['adjusted_efficiency']} km/kWh

Projected arrival SOC: {result['projected_arrival_soc']}%

Charging needed: {'Yes' if result['charging_needed'] else 'No'}

{station_text}

Recommendation: {recommendation}

Drive safe. Don't text and drive.

{disclaimer}
"""

    return response

**Demo 1**

The code predicts battery usage and trip feasibility for a highway drive to Clark with the BOSS persona. Code returns how the system communicates driving insights in a structured format.

In [38]:
trip1 = predict_trip(
    distance_km=95,
    soc=40,
    environment="highway"
)

print(persona_response("boss", "Clark", trip1))


System analysis complete.

Destination: Clark

Adjusted efficiency: 4.46 km/kWh

Projected arrival SOC: -0.94%

Charging needed: Yes



Recommendation: Charging is recommended before or during the trip.

Drive safe. Don't text and drive.


Demo Disclaimer:
This prototype assumes clear weather conditions (no rain).
Future versions would integrate real-time weather APIs.
Results may vary during rainy conditions.




**Demo 2**

Code demonstrates trip to Baguio. In addition to predicting battery usage, the model recommends a charging station from the dataset uploaded. This demo shows how the driving assistant can combine analytics and location based data to provide actionable insights.

In [39]:
trip2 = predict_trip(
    distance_km=245,
    soc=60,
    environment="mountain"
)

stations = recommend_chargers("benguet", charging_stations)

best_station = stations.iloc[0]

print(persona_response(
    "sandy",
    "Baguio",
    trip2,
    best_station
))


Rise and conquer, you little ray of sunshine.

Destination: Baguio

Adjusted efficiency: 2.62 km/kWh

Projected arrival SOC: -119.49%

Charging needed: Yes

Recommended charging station: Baguio Country Club (Baguio, benguet)

Recommendation: Charging is recommended before or during the trip.

Drive safe. Don't text and drive.


Demo Disclaimer:
This prototype assumes clear weather conditions (no rain).
Future versions would integrate real-time weather APIs.
Results may vary during rainy conditions.




**Demo 3**

Code shows a demonstration for a trip to Tagaytay. Carlo persona is used to present the results in a more informative and conversational style.

In [32]:
trip3 = predict_trip(
    distance_km=70,
    soc=55,
    environment="city"
)

print(persona_response("carlo","Tagaytay",trip3))


Fun fact: EVs convert most electricity directly into motion.

Destination: Tagaytay

Adjusted efficiency: 5.78 km/kWh

Projected arrival SOC: 31.69%

Charging needed: No



Recommendation: Proceed with the trip.

Drive safe. Don't text and drive.


⚠ Demo Disclaimer:
This prototype assumes clear weather conditions (no rain).
Future versions would integrate real-time weather APIs.
Results may vary during rainy conditions.




**Charging Station List**

In [33]:
stations = recommend_chargers("metro manila", charging_stations)

stations

,Charging Station Name,City,Province,Charger Type,ChargePoint Count,Rate (PHP/kWh)
0,Aurelia Residences,Taguig,metro manila,AC,1,22.0
1,Aurelia Residences,Taguig,metro manila,DC,1,31.0
2,EVOxCharge - In.hub Station,Taguig,metro manila,AC,6,24.0
